# Chess Vision Coach — YOLOv8m Fine-tuning

Fine-tunes the existing `yolov8m-chess.onnx` model on real Staunton tournament photos
from the `samryan18/chess-dataset`.

**Runtime:** GPU (T4 is fine). Runtime → Change runtime type → T4 GPU.

**Time:** ~60–90 minutes for 50 epochs on 400 images.

**Output:** a new `yolov8m-chess-finetuned.onnx` you drop into `chess_vision/models/`.

## Step 1 — Upload your dataset

Run `prepare_finetune_dataset.py` locally first, then zip and upload:

```bash
# On your machine:
cd chess-vision-coach
python prepare_finetune_dataset.py
zip -r finetune_data.zip finetune_data/
```

Then upload `finetune_data.zip` using the Files panel on the left (📁 icon).

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), 'No GPU found — change runtime to T4!'
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q ultralytics

In [ ]:
import zipfile, pathlib

# Unzip the uploaded dataset
with zipfile.ZipFile('finetune_data.zip') as z:
    z.extractall('.')

data_dir = pathlib.Path('finetune_data')
assert data_dir.exists(), 'finetune_data/ not found — did you upload finetune_data.zip?'

n_train = len(list((data_dir / 'images' / 'train').glob('*.jpg')))
n_val   = len(list((data_dir / 'images' / 'val').glob('*.jpg')))
print(f'Train: {n_train} images, Val: {n_val} images')

In [ ]:
# Patch the dataset.yaml to use the Colab absolute path
import yaml, pathlib

yaml_path = pathlib.Path('finetune_data/dataset.yaml')
cfg = yaml.safe_load(yaml_path.read_text())
cfg['path'] = str(pathlib.Path('finetune_data').resolve())
yaml_path.write_text(yaml.dump(cfg, default_flow_style=False))
print('dataset.yaml updated:')
print(yaml_path.read_text())

## Step 2 — Fine-tune

Starts from the official `yolov8m.pt` weights (ImageNet pre-trained),
fine-tunes on the chess dataset for 50 epochs.

Why not start from our existing ONNX? Ultralytics trains from `.pt` checkpoints,
not ONNX. The `yolov8m.pt` starting point is very good — chess pieces are objects
with clear boundaries, exactly what YOLO is pre-trained to detect.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # downloads ~52MB pre-trained weights

results = model.train(
    data='finetune_data/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,           # fits T4 16GB; lower to 8 if OOM
    workers=2,
    patience=15,        # early stop if val mAP doesn't improve for 15 epochs
    lr0=0.001,          # lower LR for fine-tuning (default is 0.01)
    lrf=0.01,
    warmup_epochs=3,
    augment=True,
    # Augmentations tuned for top-down chess photos:
    degrees=10,         # slight rotation (camera tilt)
    flipud=0.0,         # never flip vertically (board orientation matters)
    fliplr=0.5,         # horizontal flip is fine (symmetric)
    hsv_h=0.015,        # small hue shift (lighting variation)
    hsv_s=0.5,
    hsv_v=0.4,
    project='chess_finetune',
    name='yolov8m_chess',
    exist_ok=True,
)

In [ ]:
# Show training curves
from IPython.display import Image
Image('chess_finetune/yolov8m_chess/results.png')

## Step 3 — Export to ONNX

The class names must be embedded in the ONNX metadata so `ModelBackend`
auto-configures correctly (no code changes needed).

In [ ]:
best_pt = 'chess_finetune/yolov8m_chess/weights/best.pt'
best_model = YOLO(best_pt)

# Export to ONNX with metadata
best_model.export(
    format='onnx',
    imgsz=640,
    opset=12,      # wide compatibility with onnxruntime 1.x
    simplify=True,
)
print('Exported: chess_finetune/yolov8m_chess/weights/best.onnx')

## Step 4 — Quick validation before downloading

Run the model on a few val images to sanity-check detections.

In [ ]:
import pathlib
from ultralytics import YOLO
from IPython.display import Image as IPImage

best_model = YOLO('chess_finetune/yolov8m_chess/weights/best.pt')

val_imgs = sorted(pathlib.Path('finetune_data/images/val').glob('*.jpg'))[:4]
preds = best_model.predict(val_imgs, conf=0.25, save=True,
                            project='debug_preds', name='val', exist_ok=True)

for img_path in sorted(pathlib.Path('debug_preds/val').glob('*.jpg'))[:2]:
    print(img_path.name)
    display(IPImage(str(img_path)))

## Step 5 — Download and deploy

Download the ONNX file from Colab and drop it into `chess_vision/models/`.

In [ ]:
from google.colab import files
files.download('chess_finetune/yolov8m_chess/weights/best.onnx')

## Deploy

```bash
# Rename and drop into the models directory:
mv best.onnx chess-vision-coach/chess_vision/models/yolov8m-chess.onnx

# Restart the server:
CVC_BACKEND=model ./run.sh
```

The `ModelBackend` auto-reads the ONNX metadata — no code changes needed.

Run the benchmark to confirm improvement:
```bash
python bench/benchmark.py --n 30
```

**Expected after fine-tuning:** recall >70%, type accuracy >60%, exact match >50%.